In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
from itertools import combinations

In [2]:
cd ..

/ewsc/exxact04/vshir/tcga


In [3]:
df_1 = pd.read_pickle('pkl/cluster_50_perm_v3_pt_1.pkl') 
df_2 = pd.read_pickle('pkl/cluster_50_perm_v3_pt_2.pkl') 
df_3 = pd.read_pickle('pkl/cluster_50_perm_v3_pt_3.pkl') 
df_4 = pd.read_pickle('pkl/cluster_50_perm_v3_pt_4.pkl') 
df_5 = pd.read_pickle('pkl/cluster_50_perm_v3_pt_5.pkl') 

In [4]:
# 1. Store your dataframes in a list for easy iteration
dfs = [df_1, df_2, df_3, df_4, df_5]

# 2. Extract only the frequency columns (freq_0 to freq_99) from each dataframe
# We use .filter(like='freq_') to grab any column starting with that prefix
freq_parts = [df.filter(like='freq_') for df in dfs]

# 3. Concatenate them horizontally
# axis=1 means side-by-side
combined_freqs = pd.concat(freq_parts, axis=1)

# 4. Renumber the columns from freq_0 to freq_499
combined_freqs.columns = [f'freq_{i}' for i in range(combined_freqs.shape[1])]

# 5. Take the metadata columns from the first dataframe and join with the new frequencies
metadata = df_1[['cancer_type', 'cluster 1', 'cluster 2']]
final_df = pd.concat([metadata, combined_freqs], axis=1)

In [5]:
final_df

,cancer_type,cluster 1,cluster 2,freq_0,freq_1,freq_2,freq_3,freq_4,freq_5,freq_6,...,freq_490,freq_491,freq_492,freq_493,freq_494,freq_495,freq_496,freq_497,freq_498,freq_499
0,coad,0,1,0.001407,0.001412,0.001409,0.001389,0.001403,0.001427,0.001401,...,0.001405,0.001376,0.001400,0.001415,0.001403,0.001432,0.001402,0.001388,0.001373,0.001394
1,coad,0,2,0.014688,0.014624,0.014757,0.014747,0.014699,0.014757,0.014706,...,0.014716,0.014765,0.014588,0.014725,0.014673,0.014741,0.014690,0.014752,0.014670,0.014659
2,coad,0,3,0.000018,0.000021,0.000019,0.000024,0.000019,0.000021,0.000021,...,0.000023,0.000019,0.000016,0.000022,0.000020,0.000021,0.000023,0.000020,0.000017,0.000021
3,coad,0,4,0.000031,0.000030,0.000026,0.000031,0.000030,0.000027,0.000037,...,0.000035,0.000033,0.000030,0.000030,0.000038,0.000036,0.000036,0.000032,0.000033,0.000029
4,coad,0,5,0.001303,0.001356,0.001298,0.001311,0.001305,0.001305,0.001301,...,0.001344,0.001297,0.001337,0.001322,0.001320,0.001309,0.001314,0.001281,0.001287,0.001322
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6120,read,46,48,0.226742,0.224056,0.222908,0.222454,0.221194,0.222842,0.226086,...,0.223489,0.225180,0.222627,0.221911,0.223215,0.224007,0.225832,0.223565,0.226121,0.227553
6121,read,46,49,0.451414,0.452609,0.450572,0.455307,0.451835,0.450120,0.452173,...,0.451418,0.451304,0.454541,0.448760,0.452442,0.452992,0.450697,0.452736,0.450086,0.448571
6122,read,47,48,0.426306,0.424556,0.420188,0.421746,0.425118,0.420463,0.427921,...,0.425739,0.426724,0.429005,0.423517,0.425033,0.423068,0.427559,0.421449,0.422530,0.425644
6123,read,47,49,0.573694,0.575444,0.579812,0.578254,0.574882,0.579537,0.572079,...,0.574261,0.573276,0.570995,0.576483,0.574967,0.576932,0.572441,0.578551,0.577470,0.574356


In [6]:
noisy_clusters = [0,3,7,21,27,34,45,46] #filter out the noisy clusters 
df = final_df[(~final_df['cluster 1'].isin(noisy_clusters)) & (~final_df['cluster 2'].isin(noisy_clusters))] #remove self interaction and noise 
df

,cancer_type,cluster 1,cluster 2,freq_0,freq_1,freq_2,freq_3,freq_4,freq_5,freq_6,...,freq_490,freq_491,freq_492,freq_493,freq_494,freq_495,freq_496,freq_497,freq_498,freq_499
49,coad,1,2,0.011992,0.011757,0.011859,0.011594,0.011883,0.011968,0.011911,...,0.011766,0.011858,0.011635,0.011853,0.011685,0.011786,0.011837,0.011562,0.011817,0.011821
51,coad,1,4,0.000142,0.000159,0.000153,0.000116,0.000168,0.000150,0.000142,...,0.000139,0.000139,0.000108,0.000145,0.000139,0.000165,0.000170,0.000156,0.000159,0.000162
52,coad,1,5,0.005128,0.005306,0.005347,0.005491,0.005523,0.005212,0.005300,...,0.005124,0.005340,0.005343,0.005293,0.005241,0.005193,0.005314,0.005384,0.005103,0.005316
53,coad,1,6,0.034859,0.034038,0.034653,0.034931,0.034258,0.034688,0.034211,...,0.034111,0.034570,0.034605,0.034656,0.033933,0.033772,0.034546,0.034355,0.034275,0.034388
55,coad,1,8,0.002384,0.002237,0.002438,0.002385,0.002357,0.002356,0.002346,...,0.002276,0.002461,0.002442,0.002354,0.002330,0.002400,0.002326,0.002428,0.002452,0.002506
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,read,44,48,0.414988,0.419143,0.413194,0.412882,0.414259,0.412174,0.413989,...,0.419355,0.419192,0.421821,0.416059,0.419228,0.415796,0.411943,0.416572,0.416396,0.418289
6114,read,44,49,0.153874,0.154673,0.155302,0.152795,0.153355,0.155312,0.156169,...,0.152422,0.154283,0.151069,0.153971,0.153765,0.154339,0.154815,0.155216,0.152106,0.154387
6122,read,47,48,0.426306,0.424556,0.420188,0.421746,0.425118,0.420463,0.427921,...,0.425739,0.426724,0.429005,0.423517,0.425033,0.423068,0.427559,0.421449,0.422530,0.425644
6123,read,47,49,0.573694,0.575444,0.579812,0.578254,0.574882,0.579537,0.572079,...,0.574261,0.573276,0.570995,0.576483,0.574967,0.576932,0.572441,0.578551,0.577470,0.574356


In [7]:
pairwise_summary_df = pd.read_pickle('pkl/cluster_50_pairwise_summary_04_06_26.pkl') #summary of CN1 CN2 frequencies for each cancer type 
pairwise_summary_df = pairwise_summary_df[(~pairwise_summary_df['cluster 1'].isin(noisy_clusters)) & (~pairwise_summary_df['cluster 2'].isin(noisy_clusters))]
pairwise_summary_df = pairwise_summary_df.sort_values(['cancer_type','cluster 1', 'cluster 2'])
pairwise_summary_df

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency
49,coad,1,2,3739,43957,351321,0.010643
51,coad,1,4,91,10745,85876,0.001060
52,coad,1,5,2228,36195,289237,0.007703
53,coad,1,6,4471,35829,286306,0.015616
55,coad,1,8,1612,19618,156871,0.010276
...,...,...,...,...,...,...,...
5807,stad,44,48,3816,63961,505297,0.007552
5808,stad,44,49,39,17637,139913,0.000279
5816,stad,47,48,12384,85161,647334,0.019131
5817,stad,47,49,5744,143759,1106262,0.005192


In [8]:
# 1. Perform a left merge with an indicator
comparison_df = pd.merge(
    df[['cancer_type', 'cluster 1', 'cluster 2']], 
    pairwise_summary_df[['cancer_type', 'cluster 1', 'cluster 2']], 
    on=['cancer_type', 'cluster 1', 'cluster 2'], 
    how='left', 
    indicator=True
)

# 2. Filter for rows that only exist in final_df
missing_in_summary = comparison_df[comparison_df['_merge'] == 'left_only']

# 3. View the results
print(f"Number of missing cluster pairs: {len(missing_in_summary)}")
missing_in_summary.head()

Number of missing cluster pairs: 179


,cancer_type,cluster 1,cluster 2,_merge
89,coad,4,14,left_only
104,coad,4,31,left_only
107,coad,4,35,left_only
200,coad,8,14,left_only
270,coad,10,15,left_only


In [9]:
# 1. Align the data (ensure we only test pairs present in both)
merged = pd.merge(
    pairwise_summary_df, 
    df, 
    on=['cancer_type', 'cluster 1', 'cluster 2'], 
    how='inner'
)
n = 500
perm_cols = [c for c in merged.columns if c.startswith("freq_")]
merged["null_mean"] = merged[perm_cols].mean(axis=1)
merged['null_min'] = merged[perm_cols].min(axis=1)
merged['null_max'] = merged[perm_cols].max(axis=1) 
merged['null_diff'] = merged['null_max']-merged['null_min'] 
merged['null_range'] = (merged['frequency'] >= merged['null_min']) & (merged['frequency'] <= merged['null_max'])
merged

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_495,freq_496,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range
0,coad,1,2,3739,43957,351321,0.010643,0.011992,0.011757,0.011859,...,0.011786,0.011837,0.011562,0.011817,0.011821,0.011810,0.011038,0.012380,0.001342,False
1,coad,1,4,91,10745,85876,0.001060,0.000142,0.000159,0.000153,...,0.000165,0.000170,0.000156,0.000159,0.000162,0.000149,0.000094,0.000204,0.000110,False
2,coad,1,5,2228,36195,289237,0.007703,0.005128,0.005306,0.005347,...,0.005193,0.005314,0.005384,0.005103,0.005316,0.005317,0.004916,0.005599,0.000683,False
3,coad,1,6,4471,35829,286306,0.015616,0.034859,0.034038,0.034653,...,0.033772,0.034546,0.034355,0.034275,0.034388,0.034320,0.033393,0.035128,0.001735,False
4,coad,1,8,1612,19618,156871,0.010276,0.002384,0.002237,0.002438,...,0.002400,0.002326,0.002428,0.002452,0.002506,0.002365,0.002069,0.002592,0.000522,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4121,stad,44,48,3816,63961,505297,0.007552,0.139063,0.139639,0.140011,...,0.139630,0.139632,0.139095,0.139804,0.138591,0.139274,0.136292,0.141657,0.005364,False
4122,stad,44,49,39,17637,139913,0.000279,0.088319,0.088235,0.088609,...,0.088516,0.088081,0.087516,0.087985,0.088892,0.088134,0.086536,0.089991,0.003455,False
4123,stad,47,48,12384,85161,647334,0.019131,0.557531,0.556417,0.560339,...,0.554763,0.557529,0.557111,0.558906,0.555485,0.556855,0.549405,0.562845,0.013440,False
4124,stad,47,49,5744,143759,1106262,0.005192,0.442469,0.443583,0.439661,...,0.445237,0.442471,0.442889,0.441094,0.444515,0.443145,0.437155,0.450595,0.013440,False


In [10]:
merged.null_range.value_counts()

null_range
False    4014
True      112
Name: count, dtype: int64

In [11]:
filtered_df = merged[merged['contact count']>100].copy()
filtered_df 

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_495,freq_496,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range
0,coad,1,2,3739,43957,351321,0.010643,0.011992,0.011757,0.011859,...,0.011786,0.011837,0.011562,0.011817,0.011821,0.011810,0.011038,0.012380,0.001342,False
2,coad,1,5,2228,36195,289237,0.007703,0.005128,0.005306,0.005347,...,0.005193,0.005314,0.005384,0.005103,0.005316,0.005317,0.004916,0.005599,0.000683,False
3,coad,1,6,4471,35829,286306,0.015616,0.034859,0.034038,0.034653,...,0.033772,0.034546,0.034355,0.034275,0.034388,0.034320,0.033393,0.035128,0.001735,False
4,coad,1,8,1612,19618,156871,0.010276,0.002384,0.002237,0.002438,...,0.002400,0.002326,0.002428,0.002452,0.002506,0.002365,0.002069,0.002592,0.000522,False
5,coad,1,9,33960,40767,325839,0.104223,0.057725,0.057318,0.056930,...,0.057432,0.056378,0.056822,0.056948,0.056497,0.057003,0.056021,0.058066,0.002044,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4120,stad,44,47,5091,190414,1506346,0.003380,0.155147,0.155014,0.155104,...,0.154534,0.155233,0.155559,0.155553,0.157102,0.155240,0.152393,0.157886,0.005493,False
4121,stad,44,48,3816,63961,505297,0.007552,0.139063,0.139639,0.140011,...,0.139630,0.139632,0.139095,0.139804,0.138591,0.139274,0.136292,0.141657,0.005364,False
4123,stad,47,48,12384,85161,647334,0.019131,0.557531,0.556417,0.560339,...,0.554763,0.557529,0.557111,0.558906,0.555485,0.556855,0.549405,0.562845,0.013440,False
4124,stad,47,49,5744,143759,1106262,0.005192,0.442469,0.443583,0.439661,...,0.445237,0.442471,0.442889,0.441094,0.444515,0.443145,0.437155,0.450595,0.013440,False


In [12]:
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multitest import fdrcorrection 

# Extract permutation columns as a matrix
perm_cols = [f'freq_{i}' for i in range(500)]
null_matrix = merged[perm_cols].values # Shape: (4321, 500)
obs_values = merged['frequency'].values[:, np.newaxis] # Shape: (4321, 1)

# Vectorized Two-Sided P-value calculation
n = 500
perm_cols = [c for c in merged.columns if c.startswith("freq_")]
merged["null_mean"] = merged[perm_cols].mean(axis=1)

def permutation_p(row):
    obs = row["frequency"]
    perms = row[perm_cols].values
    mu = perms.mean()

    extreme = np.sum(np.abs(perms - mu) <= abs(obs - mu))
    p = (extreme + 1) / (len(perms) + 1)

    return p

merged["p_value"] = merged.apply(permutation_p, axis=1)
merged["p_adj"] = fdrcorrection(merged["p_value"], alpha=0.05, method = 'indep')[1]

In [13]:
merged

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range,p_value,p_adj
0,coad,1,2,3739,43957,351321,0.010643,0.011992,0.011757,0.011859,...,0.011562,0.011817,0.011821,0.011810,0.011038,0.012380,0.001342,False,1.0,1.0
1,coad,1,4,91,10745,85876,0.001060,0.000142,0.000159,0.000153,...,0.000156,0.000159,0.000162,0.000149,0.000094,0.000204,0.000110,False,1.0,1.0
2,coad,1,5,2228,36195,289237,0.007703,0.005128,0.005306,0.005347,...,0.005384,0.005103,0.005316,0.005317,0.004916,0.005599,0.000683,False,1.0,1.0
3,coad,1,6,4471,35829,286306,0.015616,0.034859,0.034038,0.034653,...,0.034355,0.034275,0.034388,0.034320,0.033393,0.035128,0.001735,False,1.0,1.0
4,coad,1,8,1612,19618,156871,0.010276,0.002384,0.002237,0.002438,...,0.002428,0.002452,0.002506,0.002365,0.002069,0.002592,0.000522,False,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4121,stad,44,48,3816,63961,505297,0.007552,0.139063,0.139639,0.140011,...,0.139095,0.139804,0.138591,0.139274,0.136292,0.141657,0.005364,False,1.0,1.0
4122,stad,44,49,39,17637,139913,0.000279,0.088319,0.088235,0.088609,...,0.087516,0.087985,0.088892,0.088134,0.086536,0.089991,0.003455,False,1.0,1.0
4123,stad,47,48,12384,85161,647334,0.019131,0.557531,0.556417,0.560339,...,0.557111,0.558906,0.555485,0.556855,0.549405,0.562845,0.013440,False,1.0,1.0
4124,stad,47,49,5744,143759,1106262,0.005192,0.442469,0.443583,0.439661,...,0.442889,0.441094,0.444515,0.443145,0.437155,0.450595,0.013440,False,1.0,1.0


In [14]:
p_val_df = merged[merged['p_value']<0.05]
p_val_df

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range,p_value,p_adj
370,coad,13,32,36201,195546,1509190,0.023987,0.023944,0.024213,0.023714,...,0.024284,0.024023,0.023987,0.023989,0.023619,0.024470,0.000850,True,0.015968,1.0
2577,read,4,38,89,16501,130989,0.000679,0.000617,0.000826,0.000655,...,0.000635,0.000618,0.000685,0.000680,0.000527,0.000850,0.000322,True,0.013972,1.0
2987,read,20,28,1,39,312,0.003205,0.003727,0.002609,0.003736,...,0.004049,0.003212,0.004139,0.003168,0.001447,0.005234,0.003787,True,0.039920,1.0


In [15]:
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.multitest import fdrcorrection 

# Extract permutation columns as a matrix
perm_cols = [f'freq_{i}' for i in range(500)]
null_matrix = filtered_df[perm_cols].values # Shape: (4321, 500)
obs_values = filtered_df['frequency'].values[:, np.newaxis] # Shape: (4321, 1)

# Vectorized Two-Sided P-value calculation
n = 500
perm_cols = [c for c in filtered_df.columns if c.startswith("freq_")]
filtered_df["null_mean"] = filtered_df[perm_cols].mean(axis=1)

def permutation_p(row):
    obs = row["frequency"]
    perms = row[perm_cols].values
    mu = perms.mean()

    extreme = np.sum(np.abs(perms - mu) <= abs(obs - mu))
    p = (extreme + 1) / (len(perms) + 1)

    return p

filtered_df["p_value"] = filtered_df.apply(permutation_p, axis=1)
filtered_df["p_adj"] = fdrcorrection(filtered_df["p_value"], alpha=0.05, method = 'indep')[1]

In [16]:
filtered_df

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range,p_value,p_adj
0,coad,1,2,3739,43957,351321,0.010643,0.011992,0.011757,0.011859,...,0.011562,0.011817,0.011821,0.011810,0.011038,0.012380,0.001342,False,1.0,1.0
2,coad,1,5,2228,36195,289237,0.007703,0.005128,0.005306,0.005347,...,0.005384,0.005103,0.005316,0.005317,0.004916,0.005599,0.000683,False,1.0,1.0
3,coad,1,6,4471,35829,286306,0.015616,0.034859,0.034038,0.034653,...,0.034355,0.034275,0.034388,0.034320,0.033393,0.035128,0.001735,False,1.0,1.0
4,coad,1,8,1612,19618,156871,0.010276,0.002384,0.002237,0.002438,...,0.002428,0.002452,0.002506,0.002365,0.002069,0.002592,0.000522,False,1.0,1.0
5,coad,1,9,33960,40767,325839,0.104223,0.057725,0.057318,0.056930,...,0.056822,0.056948,0.056497,0.057003,0.056021,0.058066,0.002044,False,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4120,stad,44,47,5091,190414,1506346,0.003380,0.155147,0.155014,0.155104,...,0.155559,0.155553,0.157102,0.155240,0.152393,0.157886,0.005493,False,1.0,1.0
4121,stad,44,48,3816,63961,505297,0.007552,0.139063,0.139639,0.140011,...,0.139095,0.139804,0.138591,0.139274,0.136292,0.141657,0.005364,False,1.0,1.0
4123,stad,47,48,12384,85161,647334,0.019131,0.557531,0.556417,0.560339,...,0.557111,0.558906,0.555485,0.556855,0.549405,0.562845,0.013440,False,1.0,1.0
4124,stad,47,49,5744,143759,1106262,0.005192,0.442469,0.443583,0.439661,...,0.442889,0.441094,0.444515,0.443145,0.437155,0.450595,0.013440,False,1.0,1.0


In [17]:
new_p_val_df = filtered_df[filtered_df['p_value']<0.05]
new_p_val_df

,cancer_type,cluster 1,cluster 2,contact count,total cluster 1,total_c1_interactions,frequency,freq_0,freq_1,freq_2,...,freq_497,freq_498,freq_499,null_mean,null_min,null_max,null_diff,null_range,p_value,p_adj
370,coad,13,32,36201,195546,1509190,0.023987,0.023944,0.024213,0.023714,...,0.024284,0.024023,0.023987,0.023989,0.023619,0.02447,0.00085,True,0.015968,1.0
